# Data Collection: Employee Reviews Sentiment Analysis

This notebook downloads employee reviews from Kaggle, performs initial cleaning, creates sentiment labels, and uploads the cleaned data to S3 for downstream analysis.

**Dataset**: Employee Reviews (Glassdoor/Kaggle) - ~67K reviews with star ratings (1-5), pros, cons, summary, and advice_to_management columns across major tech companies (Google, Amazon, Apple, Facebook, Microsoft, Netflix).

**Output**: Cleaned parquet saved to S3 with three-class sentiment labels (negative/neutral/positive) and binary labels (negative/positive).

## Imports

In [1]:
import kagglehub
import boto3
import os
import pandas as pd
import numpy as np
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

print('All imports successful')

All imports successful


## Constants

In [2]:
# S3 Configuration
str_bucket = 'nlp-sentiment-analysis-demo'
str_region = 'us-east-1'
str_task = 'sentiment_analysis'

# Input dataset details
str_kaggle_dataset = 'fireball684/hackerearthericsson'
str_local_filename = 'employee_reviews_raw.csv'

# Output filenames
str_output_filename = 'employee_reviews_clean.parquet'
str_s3_output_path = f'00_data_collection/{str_output_filename}'

# AWS credentials (assumes IAM role or environment variables are set)
s3_client = boto3.client('s3', region_name=str_region)

print(f'S3 Bucket: {str_bucket}')
print(f'Region: {str_region}')

S3 Bucket: nlp-sentiment-analysis-demo
Region: us-east-1


## Download Dataset from Kaggle

In [ ]:
# Download dataset from Kaggle using kagglehub
try:
    str_path = kagglehub.dataset_download(str_kaggle_dataset)
    # List all CSV files in the downloaded directory
    list_csv_files = [f for f in os.listdir(str_path) if f.endswith('.csv')]
    if len(list_csv_files) == 0:
        raise FileNotFoundError('No CSV files found in downloaded dataset')
    
    print(f'Downloaded dataset from Kaggle: {str_kaggle_dataset}')
    print(f'  Path: {str_path}')
    print(f'  CSV files found: {list_csv_files}')
    
    # Pick the data file: choose the largest CSV (skip sample_submission)
    int_best_size = 0
    str_best_file = None
    for str_f in list_csv_files:
        str_full = os.path.join(str_path, str_f)
        int_size = os.path.getsize(str_full)
        print(f'    {str_f}: {int_size / 1024:.1f} KB')
        if int_size > int_best_size:
            int_best_size = int_size
            str_best_file = str_f
    
    str_csv_path = os.path.join(str_path, str_best_file)
    df_raw = pd.read_csv(str_csv_path)
    print(f'\n  Using: {str_best_file} ({len(df_raw)} rows, {len(df_raw.columns)} columns)')
    print(f'  Columns: {list(df_raw.columns)}')
except Exception as e:
    print(f'Kaggle download failed: {e}')
    # Try loading from local file
    try:
        df_raw = pd.read_csv(str_local_filename)
        print(f'Loaded dataset from local file: {str_local_filename}')
    except FileNotFoundError:
        print(f'Local file not found either. Creating synthetic dataset for testing...')
        np.random.seed(42)
        list_companies = ['Google', 'Amazon', 'Apple', 'Facebook', 'Microsoft', 'Netflix']
        int_num_samples = 600

        list_pros = [
            'Great work-life balance and flexible hours',
            'Excellent compensation and stock options',
            'Amazing coworkers and collaborative culture',
            'Strong mentorship programs for career growth',
            'Free food and beautiful campus',
            'Cutting edge technology and interesting projects',
            'Good health benefits and retirement plans',
            'Opportunities to work on impactful products',
            'Transparent leadership and open communication',
            'Generous vacation policy and remote work options'
        ]
        list_cons = [
            'Long hours and intense deadlines',
            'High pressure to deliver results constantly',
            'Poor management and lack of direction',
            'Limited advancement opportunities in some teams',
            'Bureaucratic processes slow everything down',
            'Work-life balance is nonexistent during crunch',
            'Too many meetings and not enough coding time',
            'Compensation below market for some roles',
            'Toxic team culture in certain departments',
            'Frequent reorgs create instability and confusion'
        ]

        df_raw = pd.DataFrame({
            'company_name': np.random.choice(list_companies, int_num_samples),
            'overall_rating': np.random.choice(
                [1, 2, 3, 4, 5], int_num_samples, p=[0.10, 0.15, 0.15, 0.30, 0.30]
            ),
            'pros': np.random.choice(list_pros, int_num_samples),
            'cons': np.random.choice(list_cons, int_num_samples),
            'summary': np.random.choice([
                'Great company overall but demanding work',
                'Good place to start your career',
                'Decent pay but poor management',
                'Would not recommend to a friend',
                'Best job I have ever had',
                'Average workplace nothing special',
                'Excellent benefits but high stress',
                'Good company with room to improve'
            ], int_num_samples),
            'advice_to_management': np.random.choice([
                'Focus on work-life balance',
                'Listen to employees more',
                'Improve internal communication',
                'Stop micromanaging teams',
                'Invest in employee development'
            ], int_num_samples)
        })

print(f'\nDataset shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
print(f'\nFirst few rows:')
print(df_raw.head())

## Data Cleaning

In [ ]:
# Display initial statistics
print('=== INITIAL DATASET STATISTICS ===')
print(f'Total rows: {len(df_raw)}')
print(f'Total columns: {len(df_raw.columns)}')
print(f'\nColumn names: {list(df_raw.columns)}')
print(f'\nMissing values:')
print(df_raw.isnull().sum())
print(f'\nData types:')
print(df_raw.dtypes)
print(f'\nFirst 3 rows (all columns):')
print(df_raw.head(3).to_string())

In [ ]:
# Standardize column names
# The Kaggle dataset may use different column naming conventions.
# Map to our expected schema: company_name, overall_rating, pros, cons, summary, advice_to_management
dict_col_mapping = {}
list_raw_cols = [c.lower().strip() for c in df_raw.columns]
df_raw.columns = [c.lower().strip() for c in df_raw.columns]

# Map rating column
for str_candidate in ['overall_rating', 'overall-rating', 'overall', 'rating', 'star_rating']:
    if str_candidate in df_raw.columns:
        dict_col_mapping[str_candidate] = 'overall_rating'
        break

# Map company column
for str_candidate in ['company_name', 'company', 'firm', 'employer']:
    if str_candidate in df_raw.columns:
        dict_col_mapping[str_candidate] = 'company_name'
        break

# Map text columns
for str_target, list_candidates in {
    'pros': ['pros', 'pro', 'positive', 'positives'],
    'cons': ['cons', 'con', 'negative', 'negatives'],
    'summary': ['summary', 'headline', 'title', 'review_title'],
    'advice_to_management': ['advice_to_management', 'advice-to-mgmt', 'advice', 'management_advice']
}.items():
    for str_candidate in list_candidates:
        if str_candidate in df_raw.columns:
            dict_col_mapping[str_candidate] = str_target
            break

df_clean = df_raw.rename(columns=dict_col_mapping)

print('Column mapping applied:')
for str_old, str_new in dict_col_mapping.items():
    if str_old != str_new:
        print(f'  {str_old} -> {str_new}')
if not any(k != v for k, v in dict_col_mapping.items()):
    print('  (no renaming needed)')
print(f'\nFinal columns: {list(df_clean.columns)}')

# Verify required columns exist
list_required = ['overall_rating']
list_text_cols = [c for c in ['pros', 'cons', 'summary'] if c in df_clean.columns]
if len(list_text_cols) == 0:
    raise KeyError(f'No text columns (pros, cons, summary) found. Available columns: {list(df_clean.columns)}')

for str_col in list_required:
    if str_col not in df_clean.columns:
        raise KeyError(f'Required column "{str_col}" not found. Available columns: {list(df_clean.columns)}')

# Drop rows where rating or key text columns are missing
int_initial_rows = len(df_clean)
list_dropna_cols = ['overall_rating'] + list_text_cols
df_clean = df_clean.dropna(subset=list_dropna_cols)
int_rows_removed = int_initial_rows - len(df_clean)

print(f'\nRows removed due to missing values in {list_dropna_cols}: {int_rows_removed}')
print(f'Rows remaining: {len(df_clean)}')

In [ ]:
# Combine available text columns into a single review_text field
list_text_cols_available = [c for c in ['pros', 'cons', 'summary'] if c in df_clean.columns]
print(f'Text columns used for review_text: {list_text_cols_available}')

df_clean['review_text'] = df_clean[list_text_cols_available].astype(str).agg(' '.join, axis=1).str.strip()

# Remove rows with empty review text
df_clean = df_clean[df_clean['review_text'].str.len() > 5]
print(f'Rows after text length filtering: {len(df_clean)}')

# Ensure overall_rating is numeric
df_clean['overall_rating'] = pd.to_numeric(df_clean['overall_rating'], errors='coerce')
df_clean = df_clean.dropna(subset=['overall_rating'])
df_clean['overall_rating'] = df_clean['overall_rating'].astype(int)

# If company_name doesn't exist, fill with 'Unknown'
if 'company_name' not in df_clean.columns:
    df_clean['company_name'] = 'Unknown'
    print('Note: No company column found, defaulting to "Unknown"')

print(f'\nFinal cleaned data shape: {df_clean.shape}')

## Create Sentiment Labels

In [ ]:
# Create 3-class sentiment labels
# 1-2 stars = Negative (0)
# 3 stars = Neutral (1)
# 4-5 stars = Positive (2)
def map_to_sentiment_multiclass(int_rating):
    if int_rating <= 2:
        return 0  # Negative
    elif int_rating == 3:
        return 1  # Neutral
    else:  # 4-5
        return 2  # Positive

df_clean['sentiment_3class'] = df_clean['overall_rating'].apply(map_to_sentiment_multiclass)

print('Three-class sentiment distribution:')
print(df_clean['sentiment_3class'].value_counts().sort_index())
print(f'\nPercentages:')
print(df_clean['sentiment_3class'].value_counts(normalize=True).sort_index() * 100)

In [ ]:
# Create binary sentiment labels
# 1-3 stars = Negative (0)
# 4-5 stars = Positive (1)
def map_to_sentiment_binary(int_rating):
    if int_rating <= 3:
        return 0  # Negative
    else:
        return 1  # Positive

df_clean['sentiment_binary'] = df_clean['overall_rating'].apply(map_to_sentiment_binary)

print('Binary sentiment distribution:')
print(df_clean['sentiment_binary'].value_counts().sort_index())
print(f'\nPercentages:')
print(df_clean['sentiment_binary'].value_counts(normalize=True).sort_index() * 100)

In [ ]:
# Select and reorder columns for final dataset
list_cols = ['company_name', 'overall_rating', 'review_text', 'sentiment_3class', 'sentiment_binary']
df_clean = df_clean[list_cols]

print('Final dataset columns:')
print(df_clean.columns.tolist())
print(f'\nFinal dataset shape: {df_clean.shape}')
print(f'\nFirst 5 rows:')
print(df_clean.head())

## Upload to S3

In [ ]:
# Convert DataFrame to parquet and upload to S3
buffer = BytesIO()
df_clean.to_parquet(buffer, index=False, engine='pyarrow')
buffer.seek(0)

s3_client.put_object(
    Bucket=str_bucket,
    Key=str_s3_output_path,
    Body=buffer.getvalue(),
    ContentType='application/octet-stream'
)
str_s3_uri = f's3://{str_bucket}/{str_s3_output_path}'
print(f'Successfully uploaded cleaned data to S3')
print(f'  S3 URI: {str_s3_uri}')

## Summary Statistics

In [ ]:
print('\n' + '='*60)
print('DATA COLLECTION SUMMARY')
print('='*60)
print(f'\nDataset: Employee Reviews (Glassdoor/Kaggle)')
print(f'Total reviews processed: {len(df_clean):,}')
print(f'\nColumns: {list_cols}')
print(f'\nRating distribution:')
print(df_clean['overall_rating'].value_counts().sort_index())
print(f'\nCompanies in dataset: {df_clean["company_name"].nunique()}')
print(f'Company breakdown:')
print(df_clean['company_name'].value_counts())
print(f'\nAverage review length: {df_clean["review_text"].str.len().mean():.0f} characters')
print(f'\nThree-class label distribution:')
dict_sentiment_mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
for int_label, str_name in dict_sentiment_mapping.items():
    int_count = (df_clean['sentiment_3class'] == int_label).sum()
    flt_pct = (int_count / len(df_clean)) * 100
    print(f'  {str_name} (label {int_label}): {int_count:,} ({flt_pct:.1f}%)')
print(f'\nBinary label distribution:')
dict_binary_mapping = {0: 'Negative', 1: 'Positive'}
for int_label, str_name in dict_binary_mapping.items():
    int_count = (df_clean['sentiment_binary'] == int_label).sum()
    flt_pct = (int_count / len(df_clean)) * 100
    print(f'  {str_name} (label {int_label}): {int_count:,} ({flt_pct:.1f}%)')
print(f'\nOutput saved to: s3://{str_bucket}/{str_s3_output_path}')
print('='*60)